# Лабораторная работа 2
## Тот же диод, сетка вдвое мельче

Повторяем работу 1, уменьшив ячейки в два раза, и сравниваем.
Смысл: решение зависит от сетки. На грубой кривые рваные, на мелкой гладкие.

Геометрия и примесь те же, меняются только размеры ячеек.

In [1]:
import sys, os
ROOT = os.path.abspath(".")
while not os.path.isdir(os.path.join(ROOT, "tools")) and ROOT != "/":
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, os.path.join(ROOT, "tools"))

os.makedirs(os.path.join(ROOT, "Lab2_diode_2D_mesh", "ise"), exist_ok=True)
os.chdir(os.path.join(ROOT, "Lab2_diode_2D_mesh", "ise"))

import xp, put, dfise, viz
import numpy as np

WORK = r"C:\tcad\LAB2"     # папка внутри Windows
print(os.getcwd())

/Users/phoenix/Documents/MIET/labs_modelling/work/Lab2


## Где считаем

Ниже — вывод самих программ ISE. По нему видно версию Windows,
имя машины, версию решателя с копирайтом разработчика.

In [2]:
print(xp.run(["ver", "hostname", "echo %ISEROOT% %ISERELEASE%"]))

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>ver

Microsoft Windows XP [Версия 5.1.2600]

C:\Documents and Settings\Легошин>hostname
win-xp

C:\Documents and Settings\Легошин>echo %ISEROOT% %ISERELEASE%
C:\ISE 7.0

C:\Documents and Settings\Легошин>


In [3]:
# решатель сам печатает версию, разработчика, ОС и машину
print(xp.run([r"C:\ISE\bin\dessis -h"])[:1600])

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>C:\ISE\bin\dessis -h
*******************************************************************
                      DESSIS - version 7.0.6
              (compiled Wed Feb 28 16:33:09  2001)

   Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG
                        Zurich, Switzerland
*******************************************************************

	Running on machine with the following configuration:
	Host Name: WIN-XP
	Operating System: WindowsNT rel. 4.0 ver. 0
	Machine Type: ix86
	Date: Tue Sep 29 09:55:09 2026  (╠юёъютёъюх тЁхь (чшьр))

Usage: dessis [ options ] [ filename[.cmd] ]
Options are:
-c                    write current and acextraction files in old format
-d         

In [6]:
# ---- ВАРИАНТ ----
VARIANT = 2    # номер
d       = 0.8    # толщина диода по Oy, мкм
NB      = 1.4e+17    # бор, 1/см3
NAs     = 1.0e+17    # мышьяк, 1/см3
# -----------------
assert None not in (VARIANT, d, NB, NAs), "не заполнен вариант"

# в методичке d = 2 мкм. всё по Oy делим на k, по Ox не трогаем
k = 2.0 / d
h = d / 2.0
print(VARIANT, d, k)

2 0.8 2.5


## 1. Структура

Та же, что в работе 1.

In [7]:
bnd = f"""Silicon "SiW"
{{
\trectangle [(-1.5,{-h}) (1.5,{h})]
}}

Contact "Anode"
{{
\tline[(-0.5,{-h}) (0.5,{-h})]
}}

Contact "Cathode"
{{
\tline[(-1.5,{h}) (1.5,{h})]
}}
"""
open("sh1.bnd","w").write(bnd)
print("ок")

ок


## 2. Сетка

`s = 0.5` — это и есть «вдвое мельче». Но мельче не всё подряд: задание требует
уменьшить ячейки **вдоль Oy**, и эталонные картинки это подтверждают. По Ox у
первой сетки краевые ячейки остаются 0.5 и в этой работе — на картинках
в `Variants_1-2_2023.doc` их видно, они не поделились.

Мельче становятся: у первой сетки шаг по Oy, у второй — шаг по Ox.
Шаг второй сетки по Oy трогать нельзя, иначе у анода вылезают лишние слои,
которых в эталоне нет.

Про то, почему `MaxElementSize` — это потолок, а не размер ячейки,
написано в блокноте первой работы.

In [ ]:
s = 0.5

cmd = f"""Title "Sharp Diode"

Definitions {{
    Refinement "Mesh1"
    {{
      MaxElementSize = (0.5 {0.5*s/k:g})
      MinElementSize = (0.1 {0.1*s/k:g})
      RefineFunction = MaxTransDiff(Variable = "DopingConcentration", Value = 1)
    }}
    Refinement "Mesh2"
    {{
      MaxElementSize = ({0.25*s:g} {0.25/k:g})
      MinElementSize = ({0.05*s:g} {0.05/k:g})
    }}
    Constant "BW"  {{ Species = "BoronActiveConcentration"   Value = {NB} }}
    Constant "AsW" {{ Species = "ArsenicActiveConcentration" Value = {NAs} }}
}}

Placements {{
    Refinement "Mesh1" {{ Reference = "Mesh1"
      RefineWindow = rectangle [( -1.5 {-h} ) , ( 1.5 {h} )] }}
    Refinement "Mesh2" {{ Reference = "Mesh2"
      RefineWindow = rectangle [( -1.0 {-h} ) , ( 1.0 {-h/2} )] }}
    Constant "BW"  {{ Reference = "BW"
      EvaluateWindow {{ Element = rectangle [( -1.5 {-h} ) , ( 1.5 0 )] DecayLength = 0 }} }}
    Constant "AsW" {{ Reference = "AsW"
      EvaluateWindow {{ Element = rectangle [( -1.5 0 ) , ( 1.5 {h} )] DecayLength = 0 }} }}
}}
"""
open("sh1.cmd","w").write(cmd)
open("cut.txt","w").write(f"0.0\n{-h}\n0.0\n{h}\n")
print(cmd)

In [9]:
# файл решателя берём из работы 1 без изменений
LAB1 = os.path.join(ROOT, "Lab1_diode_2D", "ise")
open("sh1_des.cmd","w").write(open(os.path.join(LAB1,"sh1_des.cmd")).read())
print("ок")

ок


`xp` — самописный, `tools/xp.py`. ISE стоит в виртуалке с Windows XP,
ноутбук на маке. `xp` ходит туда по telnet и гоняет команды.
Ждёт приглашение `>` перед каждой следующей, иначе долгие расчёты обрываются.

- `xp.run([...])` — выполнить команды, вернуть вывод
- `xp.fetch(путь, куда)` — забрать файл из Windows
- `put.commands_for(файл, путь)` — команды заливки файла внутрь

In [10]:
# заливаем файлы в Windows
cmds = [f"mkdir {WORK}"]
for f in ("sh1.bnd", "sh1.cmd", "sh1_des.cmd", "cut.txt",):
    cmds += put.commands_for(f, f"{WORK}\\{f}")
xp.run(cmds, total=2400)
print("залито")

залито


In [11]:
# строим сетку — узлов должно стать больше, чем в работе 1
print(xp.run([f"cd /d {WORK}", r"C:\ISE\bin\mesh -t sh1"], total=1800, idle_limit=300))

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>cd /d C:\tcad\LAB2

C:\tcad\LAB2>C:\ISE\bin\mesh -t sh1
> # ISE TCAD Software Release 7.0
> # Version Mdraw 7.0.11
> # Version Mesh 7.0.11
> # Compiled Fri Feb  9 13:16:00  2001 on TERM101
> # Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG
> # ISE TCAD Software Release 7.0
> # Version Mdraw 7.0.11
> # Version Mesh 7.0.11
> # Compiled Fri Feb  9 13:16:00  2001 on TERM101
> # Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG
> Mesh version 7.0.11
> (Compiled Fri Feb  9 13:15:57  2001)
> (Compiled on TERM101)
> Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG, Zurich, Switzerland
> Running on: WindowsNT WIN-XP 4.0 0 ix86

> Process ID: 1792
>
> Flags used: -

In [12]:
# решаем
print(xp.run([f"cd /d {WORK}", r"C:\ISE\bin\dessis sh1_des.cmd"], total=3600, idle_limit=600))

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>cd /d C:\tcad\LAB2

C:\tcad\LAB2>C:\ISE\bin\dessis sh1_des.cmd
*******************************************************************
                      DESSIS - version 7.0.6
              (compiled Wed Feb 28 16:33:09  2001)

   Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG
                        Zurich, Switzerland
*******************************************************************

	Running on machine with the following configuration:
	Host Name: WIN-XP
	Operating System: WindowsNT rel. 4.0 ver. 0
	Machine Type: ix86
	Date: Tue Sep 29 09:56:35 2026  (╠юёъютёъюх тЁхь (чшьр))

Input file: sh1_des.cmd

Output file: sh1_des.log

no ACExtract file

PMIPath file: C:\ISE\tcad\7.0\

In [13]:
# делаем разрез
print(xp.run([f"cd /d {WORK}", r"C:\ISE\bin\dfisetools -plot sh1_msh.grd sh1_des.dat < cut.txt"], total=900, idle_limit=300))

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>cd /d C:\tcad\LAB2

C:\tcad\LAB2>C:\ISE\bin\dfisetools -plot sh1_msh.grd sh1_des.dat < cut.txt

1st point of line ?
    x =     y = 2nd point of line ?
    x =     y =
C:\tcad\LAB2>


## Картинки

In [14]:
# забираем результаты на мак
for f in ("sh1_msh.grd", "sh1_msh.dat", "sh1_des.dat", "sh1_des.plt", "sh1_des_plt.plt",):
    xp.fetch(f"{WORK}\\{f}", "" + f)
    print(f, os.path.getsize("" + f), "байт")

sh1_msh.grd 43365 байт
sh1_msh.dat 12798 байт
sh1_des.dat 216946 байт
sh1_des.plt 36196 байт
sh1_des_plt.plt 8097 байт


In [15]:
g2 = dfise.read_grid("sh1_msh.grd")
d2 = dfise.read_dataset("sh1_des.dat", g2)
print(len(g2["vertices"]), "узлов")

519 узлов


In [ ]:
import ise_style as ise, compare

OUT = os.path.join(ROOT, "Lab2_diode_2D_mesh", "results")
os.makedirs(OUT, exist_ok=True)
dop2 = d2["DopingConcentration"]

ise.field  (g2, d2["SpaceCharge"],            OUT + "/z2_заряд.png",
            mesh=True, log=True, doping=dop2)
ise.contour(g2, d2["ElectrostaticPotential"], OUT + "/z2_потенциал.png", doping=dop2)

c2 = viz.read_plt("sh1_des_plt.plt")
ise.xy([("SpaceCharge", c2["Distance"], c2["SpaceCharge"], "k")],
       OUT + "/z2_разрез.png")
print(sorted(os.listdir(OUT)))

### Заряд и сетка

Эталон, наша отрисовка, picasso. Разбор, чем наша отличается и как устроена
отрисовка у picasso — в блокноте первой работы.

In [ ]:
compare.trio(compare.ref(VARIANT, "z2_заряд"),
             OUT + "/z2_заряд.png", compare.pic("z2_заряд"));

### Потенциал, линии уровня

In [ ]:
compare.trio(compare.ref(VARIANT, "z2_потенциал"),
             OUT + "/z2_потенциал.png", compare.pic("z2_потенциал"));

### Заряд в сечении

In [ ]:
compare.trio(compare.ref(VARIANT, "z2_разрез"),
             OUT + "/z2_разрез.png", compare.pic("z2_разрез"), width=6.0);

## Задание 3. Сравнение сеток

Главный результат: обе кривые на одном графике.

In [ ]:
iv1 = viz.read_plt(os.path.join(LAB1, "sh1_des.plt"))
iv2 = viz.read_plt("sh1_des.plt")
srt = lambda d: (lambda V, I, o: (V[o], I[o]))(
        np.asarray(d["Anode OuterVoltage"]), np.asarray(d["Anode TotalCurrent"]),
        np.argsort(d["Anode OuterVoltage"]))
V1, I1 = srt(iv1); V2, I2 = srt(iv2)
ise.xy([("TotalCurrent", V1, I1, "k"), ("TotalCurrent", V2, I2, "r")],
       OUT + "/z3_вах.png", size=(314, 374), legend=False)
print("ток при 4 В: крупная %.5f А, мелкая %.5f А" % (I1[-1], I2[-1]))

In [ ]:
compare.trio(compare.ref(VARIANT, "z3_вах"),
             OUT + "/z3_вах.png", compare.pic("z3_вах"), width=6.0);

In [ ]:
c1 = viz.read_plt(os.path.join(LAB1, "sh1_des_plt.plt"))
ise.xy([("Potential", c1["Distance"], c1["ElectrostaticPotential"], "k"),
        ("Potential", c2["Distance"], c2["ElectrostaticPotential"], "r")],
       OUT + "/z3_потенциал.png", size=(296, 373), legend=False)

In [ ]:
compare.trio(compare.ref(VARIANT, "z3_потенциал"),
             OUT + "/z3_потенциал.png", compare.pic("z3_потенциал"), width=6.0);

## Задание 4. Без внешнего напряжения

Меняем в файле решателя одну строку: цель не 4 В, а 0 В.

In [20]:
des0 = open("sh1_des.cmd").read().replace("Voltage=4 }", "Voltage=0 }")
open("sh1_des0.cmd","w").write(des0)
for a, b in zip(open("sh1_des.cmd"), open("sh1_des0.cmd")):
    if a != b:
        print("было: ", a.strip())
        print("стало:", b.strip())

было:  Quasistationary ( MaxStep=0.01 Goal { Name="Anode" Voltage=4 } )
стало: Quasistationary ( MaxStep=0.01 Goal { Name="Anode" Voltage=0 } )


In [21]:
# отдельная папка, чтобы не затереть результаты задания 2
WORK0 = r"C:\tcad\LAB2b"
cmds = [f"mkdir {WORK0}"]
for f in ("sh1.bnd","sh1.cmd","cut.txt"):
    cmds += put.commands_for(f, f"{WORK0}\\{f}")
cmds += put.commands_for("sh1_des0.cmd", f"{WORK0}\\sh1_des.cmd")
xp.run(cmds, total=2400)
print("залито")

залито


In [22]:
# строим сетку
print(xp.run([f"cd /d {WORK0}", r"C:\ISE\bin\mesh -t sh1"], total=1800, idle_limit=300))

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>cd /d C:\tcad\LAB2b

C:\tcad\LAB2b>C:\ISE\bin\mesh -t sh1
> # ISE TCAD Software Release 7.0
> # Version Mdraw 7.0.11
> # Version Mesh 7.0.11
> # Compiled Fri Feb  9 13:16:00  2001 on TERM101
> # Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG
> # ISE TCAD Software Release 7.0
> # Version Mdraw 7.0.11
> # Version Mesh 7.0.11
> # Compiled Fri Feb  9 13:16:00  2001 on TERM101
> # Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG
> Mesh version 7.0.11
> (Compiled Fri Feb  9 13:15:57  2001)
> (Compiled on TERM101)
> Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG, Zurich, Switzerland
> Running on: WindowsNT WIN-XP 4.0 0 ix86

> Process ID: 1564
>
> Flags used:

In [23]:
# решаем при 0 В
print(xp.run([f"cd /d {WORK0}", r"C:\ISE\bin\dessis sh1_des.cmd"], total=3600, idle_limit=600))

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>cd /d C:\tcad\LAB2b

C:\tcad\LAB2b>C:\ISE\bin\dessis sh1_des.cmd
*******************************************************************
                      DESSIS - version 7.0.6
              (compiled Wed Feb 28 16:33:09  2001)

   Copyright (c) 1994-2001 ISE Integrated Systems Engineering AG
                        Zurich, Switzerland
*******************************************************************

	Running on machine with the following configuration:
	Host Name: WIN-XP
	Operating System: WindowsNT rel. 4.0 ver. 0
	Machine Type: ix86
	Date: Tue Sep 29 10:03:24 2026  (╠юёъютёъюх тЁхь (чшьр))

Input file: sh1_des.cmd

Output file: sh1_des.log

no ACExtract file

PMIPath file: C:\ISE\tcad\7.

In [24]:
# разрез
print(xp.run([f"cd /d {WORK0}", r"C:\ISE\bin\dfisetools -plot sh1_msh.grd sh1_des.dat < cut.txt"], total=900, idle_limit=300))

Welcome to Microsoft Telnet Service


login: Легошин

password:

*================================================================
Вас приветствует Telnet-сервер производства корпорации Майкрософт
*================================================================
C:\Documents and Settings\Легошин>cd /d C:\tcad\LAB2b

C:\tcad\LAB2b>C:\ISE\bin\dfisetools -plot sh1_msh.grd sh1_des.dat < cut.txt

1st point of line ?
    x =     y = 2nd point of line ?
    x =     y =
C:\tcad\LAB2b>


In [25]:
for f in ("sh1_msh.grd","sh1_des.dat","sh1_des_plt.plt"):
    xp.fetch(f"{WORK0}\\{f}", "v0_" + f)
    print("v0_" + f)

v0_sh1_msh.grd
v0_sh1_des.dat
v0_sh1_des_plt.plt


In [ ]:
g0 = dfise.read_grid("v0_sh1_msh.grd")
d0 = dfise.read_dataset("v0_sh1_des.dat", g0)
c0 = viz.read_plt("v0_sh1_des_plt.plt")

# без сетки — так в задании
ise.field(g0, d0["SpaceCharge"], OUT + "/z4_заряд.png",
          mesh=False, log=True, doping=d0["DopingConcentration"])
ise.xy([("SpaceCharge", c0["Distance"], c0["SpaceCharge"], "k")],
       OUT + "/z4_разрез.png")

In [ ]:
compare.trio(compare.ref(VARIANT, "z4_заряд"),
             OUT + "/z4_заряд.png", compare.pic("z4_заряд"));

In [ ]:
compare.trio(compare.ref(VARIANT, "z4_разрез"),
             OUT + "/z4_разрез.png", compare.pic("z4_разрез"), width=6.0);

### Вывод

_(насколько разошлись кривые и что изменилось без напряжения)_